# 09 — Evaluación Comparativa de Modelos

Evaluación final de las **cuatro arquitecturas entrenadas** sobre el **test set** de Galaxy Zoo 2 (16,670 galaxias, 6 clases).  
Cada modelo se carga desde su `best.pth` — el checkpoint con mayor val F1-macro durante el entrenamiento.

| Modelo | Notebook | Val F1 (mejor) | Época mejor / Total | Params | IMAGE_SIZE |
|---|---|---|---|---|---|
| EfficientNet-B3 | `04` | 0.6894 | 16 / 30 | ~10.7 M | 224 px |
| ResNet-50 | `05` | 0.6914 | 14 / 19 (early stop) | ~23.5 M | 224 px |
| Swin-S | `06` | 0.6962 | 25 / 30 | ~49.6 M | 308 px |
| MaxViT-T | `08` | 0.6951 | 17 / 22 (early stop) | ~30.9 M | 224 px |

> La evaluación es estrictamente sobre el **test set** — datos no vistos durante entrenamiento ni validación.  
> **Prerequisito:** ejecutar primero los notebooks `04`, `05`, `06` y `08` para generar los archivos `best.pth`.


## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez** en el entorno nuevo. Reinicia el kernel después si es la primera instalación.


In [ ]:
import subprocess, sys

# PyTorch con CUDA 12.8 (necesario para RTX 5060 Ti / Blackwell sm_120)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    '--quiet',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'Pillow', 'scikit-learn', 'tqdm',
], check=True)

print('Instalación completada. Reinicia el kernel si es la primera vez.')

## Sección 1 — Imports


In [ ]:
import gc
import os
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    efficientnet_b3, EfficientNet_B3_Weights,
    resnet50, ResNet50_Weights,
    swin_s, Swin_S_Weights,
    maxvit_t, MaxVit_T_Weights,
)
from sklearn.metrics import (
    f1_score, accuracy_score,
    classification_report, confusion_matrix,
)
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute  : sm_{cap[0]}{cap[1]}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')

## Sección 2 — Configuración


In [ ]:
# ─── Paths ───────────────────────────────────────────────────────────────────
_LOCAL     = pathlib.Path('../data')
IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
SPLITS_DIR = _LOCAL / 'splits'
CKPT_BASE  = pathlib.Path('../models/checkpoints')
LOG_DIR    = pathlib.Path('../logs')

assert IMAGES_DIR.exists(), f'IMAGES_DIR no encontrado: {IMAGES_DIR.resolve()}'
assert (SPLITS_DIR / 'test.csv').exists(), f'test.csv no encontrado: {SPLITS_DIR.resolve()}'

# ─── Clases ──────────────────────────────────────────────────────────────────
NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# ─── Configuración por modelo ─────────────────────────────────────────────────
# image_size / crop_size deben coincidir exactamente con los usados en entrenamiento.
# Swin-S usa resolución 308 px (divisible por patch_size × window_size = 28);
# los modelos CNN y MaxViT-T usan 224 px.
MODEL_CONFIGS = {
    'efficientnet_b3': {
        'label':      'EfficientNet-B3',
        'image_size': 224,
        'crop_size':  320,
        'ckpt_dir':   CKPT_BASE / 'efficientnet_b3',
        'params_m':   10.7,
        'color':      '#1f77b4',
    },
    'resnet50': {
        'label':      'ResNet-50',
        'image_size': 224,
        'crop_size':  320,
        'ckpt_dir':   CKPT_BASE / 'resnet50',
        'params_m':   23.5,
        'color':      '#ff7f0e',
    },
    'swin_s': {
        'label':      'Swin-S',
        'image_size': 308,
        'crop_size':  380,
        'ckpt_dir':   CKPT_BASE / 'swin_s',
        'params_m':   49.6,
        'color':      '#2ca02c',
    },
    'maxvit_t': {
        'label':      'MaxViT-T',
        'image_size': 224,
        'crop_size':  320,
        'ckpt_dir':   CKPT_BASE / 'maxvit_t',
        'params_m':   30.9,
        'color':      '#d62728',
    },
}

USE_AMP       = device.type == 'cuda'
BATCH_SIZE    = 64      # sin gradientes → batch mayor para evaluar más rápido
NUM_WORKERS   = 0 if os.name == 'nt' else 4
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'IMAGES_DIR : {IMAGES_DIR.resolve()}')
print(f'SPLITS_DIR : {SPLITS_DIR.resolve()}')
print(f'CKPT_BASE  : {CKPT_BASE.resolve()}')
print(f'USE_AMP    : {USE_AMP}')
print()
print('Checkpoints disponibles:')
for mname, cfg in MODEL_CONFIGS.items():
    best = cfg['ckpt_dir'] / 'best.pth'
    st   = '✓' if best.exists() else '✗  NO ENCONTRADO'
    print(f'  {cfg["label"]:<20}  best.pth {st}')

## Sección 3 — Pipeline de datos

Se crea un `DataLoader` de test independiente para cada modelo con los transforms exactos de su entrenamiento.  
`Swin-S` usa `IMAGE_SIZE=308`, `CROP_SIZE=380`; el resto usa `IMAGE_SIZE=224`, `CROP_SIZE=320`.


In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames  = df['img_filename'].to_numpy()
        self.labels     = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df; gc.collect()
        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


def make_test_loader(image_size: int, crop_size: int) -> DataLoader:
    '''DataLoader de test con transforms ajustados a cada arquitectura.'''
    tfm = transforms.Compose([
        transforms.CenterCrop(crop_size),
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return DataLoader(
        GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, tfm, CLASS_TO_IDX),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == 'cuda'),
    )


# Verificar el test set
_tmp = pd.read_csv(SPLITS_DIR / 'test.csv', usecols=['morph_label'])
print(f'Test set: {len(_tmp):,} imágenes')
print(_tmp['morph_label'].value_counts().rename('n').to_frame().to_string())
del _tmp; gc.collect()

## Sección 4 — Definición de modelos

Función `build_model` que reconstruye cada arquitectura con la cabeza personalizada de 6 clases,  
lista para cargar los pesos del checkpoint `best.pth`. No se cargan pesos preentrenados de ImageNet.


In [ ]:
def _unwrap_model(m: nn.Module) -> nn.Module:
    '''Extrae el modulo base ignorando DataParallel y torch.compile.'''
    if isinstance(m, nn.DataParallel):
        m = m.module
    if hasattr(m, '_orig_mod'):     # torch.compile() envuelve en _orig_mod
        m = m._orig_mod
    return m


def build_model(model_name: str, num_classes: int = NUM_CLASSES) -> nn.Module:
    '''Reconstruye la arquitectura con cabeza personalizada sin pesos preentrenados.'''
    if model_name == 'efficientnet_b3':
        m = efficientnet_b3(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        # classifier: [0] Dropout(0.3)  [1] Linear(1536, num_classes)

    elif model_name == 'resnet50':
        m = resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        # fc: Linear(2048, num_classes)

    elif model_name == 'swin_s':
        m = swin_s(weights=None)
        m.head = nn.Linear(m.head.in_features, num_classes)
        # head: Linear(768, num_classes)

    elif model_name == 'maxvit_t':
        m = maxvit_t(weights=None)
        m.classifier[5] = nn.Linear(m.classifier[5].in_features, num_classes)
        # classifier: [0] AvgPool  [1] Flatten  [2] LN  [3] Linear(512,512)
        #             [4] Tanh     [5] Linear(512, num_classes)

    else:
        raise ValueError(f'Arquitectura no reconocida: {model_name}')
    return m


# Verificar checkpoints y mostrar metadatos
print('Checkpoints disponibles:')
for mname, cfg in MODEL_CONFIGS.items():
    best = cfg['ckpt_dir'] / 'best.pth'
    if best.exists():
        ck = torch.load(best, map_location='cpu')
        print(f'  {cfg["label"]:<20}  epoch={ck["epoch"]+1:2d}  '
              f'val_f1={ck["best_val_f1"]:.4f}')
    else:
        print(f'  {cfg["label"]:<20}  ✗  best.pth no encontrado')

## Sección 5 — Evaluación sobre test set

Carga secuencial de cada modelo desde `best.pth` y evaluación sobre el test set.  
Cada modelo se descarga de VRAM antes de cargar el siguiente para evitar OOM.


In [ ]:
@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, use_amp: bool = USE_AMP):
    '''Pasa todo el loader por el modelo. Devuelve (preds, labels). Solo inferencia.'''
    model.eval()
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc='  Eval', unit='batch', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            logits = model(imgs)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())
    return all_preds, all_labels


print('evaluate_model definida ✓')

In [ ]:
results = {}   # model_name → dict con preds, labels, métricas, etc.

for model_name, cfg in MODEL_CONFIGS.items():
    best_ckpt = cfg['ckpt_dir'] / 'best.pth'
    if not best_ckpt.exists():
        print(f'[SKIP] {cfg["label"]} — best.pth no encontrado en {best_ckpt}')
        continue

    print(f'\n{"="*60}')
    print(f'Evaluando: {cfg["label"]}  (IMAGE_SIZE={cfg["image_size"]} px)')
    print(f'{"="*60}')

    # 1. Reconstruir modelo y cargar pesos del mejor checkpoint
    model = build_model(model_name).to(device)
    ckpt  = torch.load(best_ckpt, map_location=device)
    _unwrap_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'  Checkpoint: epoch {ckpt["epoch"]+1}  val_f1={ckpt["best_val_f1"]:.4f}')

    # 2. DataLoader con la resolución correcta para este modelo
    loader = make_test_loader(cfg['image_size'], cfg['crop_size'])

    # 3. Inferencia sobre el test set completo
    preds, labels = evaluate_model(model, loader)

    # 4. Calcular todas las métricas
    test_f1  = f1_score(labels, preds, average='macro', zero_division=0)
    test_acc = accuracy_score(labels, preds)
    report   = classification_report(
        labels, preds,
        target_names=CLASS_ORDER,
        zero_division=0,
        output_dict=True,
    )
    cm_abs   = confusion_matrix(labels, preds)
    cm_norm  = cm_abs.astype(float) / cm_abs.sum(axis=1, keepdims=True)

    results[model_name] = {
        'label':      cfg['label'],
        'preds':      preds,
        'labels':     labels,
        'f1_macro':   test_f1,
        'accuracy':   test_acc,
        'report':     report,
        'cm_abs':     cm_abs,
        'cm_norm':    cm_norm,
        'val_f1':     ckpt['best_val_f1'],
        'best_epoch': ckpt['epoch'] + 1,
        'params_m':   cfg['params_m'],
        'color':      cfg['color'],
    }

    print(f'  Test F1-macro : {test_f1:.4f}')
    print(f'  Test Accuracy : {test_acc:.4f}')

    # Liberar GPU entre modelos para evitar OOM
    del model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\n' + '='*60)
print(f'Evaluación completada: {len(results)}/{len(MODEL_CONFIGS)} modelos')

## Sección 6 — Métricas globales


In [ ]:
rows = []
for mname, r in results.items():
    rows.append({
        'Modelo':      r['label'],
        'Val F1':      r['val_f1'],
        'Test F1':     r['f1_macro'],
        'Test Acc.':   r['accuracy'],
        'Params (M)':  r['params_m'],
        'Best epoch':  r['best_epoch'],
    })

summary_df = (
    pd.DataFrame(rows)
    .sort_values('Test F1', ascending=False)
    .reset_index(drop=True)
)
summary_df.index += 1

print('\n── Comparativa global — Test set ─────────────────────────')
print(summary_df.to_string(
    float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x),
))
print()
best_row = summary_df.iloc[0]
print(f'Mejor modelo     : {best_row["Modelo"]}  (Test F1 = {best_row["Test F1"]:.4f})')
print(f'Mayor accuracy   : {summary_df.sort_values("Test Acc.", ascending=False).iloc[0]["Modelo"]}')
print(f'Más eficiente    : {summary_df.assign(ratio=summary_df["Test F1"]/summary_df["Params (M)"]).sort_values("ratio", ascending=False).iloc[0]["Modelo"]}  (mayor F1/params)')

In [ ]:
model_labels = [r['label']    for r in results.values()]
f1_vals      = [r['f1_macro'] for r in results.values()]
acc_vals     = [r['accuracy'] for r in results.values()]
colors_list  = [r['color']    for r in results.values()]
params_list  = [r['params_m'] for r in results.values()]

x = np.arange(len(model_labels))
w = 0.5

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Comparativa de modelos — Test set', fontsize=13, fontweight='bold')

# F1-macro
bars = axes[0].bar(x, f1_vals, width=w, color=colors_list, edgecolor='white', linewidth=0.8)
axes[0].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_labels, rotation=15, ha='right')
axes[0].set_ylabel('F1-macro'); axes[0].set_title('Test F1-macro')
axes[0].set_ylim(max(0, min(f1_vals) - 0.02), max(f1_vals) + 0.015)
axes[0].grid(axis='y', alpha=0.3)

# Accuracy
bars = axes[1].bar(x, acc_vals, width=w, color=colors_list, edgecolor='white', linewidth=0.8)
axes[1].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_labels, rotation=15, ha='right')
axes[1].set_ylabel('Accuracy'); axes[1].set_title('Test Accuracy')
axes[1].set_ylim(max(0, min(acc_vals) - 0.02), max(acc_vals) + 0.015)
axes[1].grid(axis='y', alpha=0.3)

# F1 vs params scatter
axes[2].scatter(params_list, f1_vals, c=colors_list, s=180, zorder=3,
                edgecolors='white', linewidths=1.0)
for lbl, px, fy in zip(model_labels, params_list, f1_vals):
    axes[2].annotate(lbl, (px, fy), textcoords='offset points', xytext=(6, 4), fontsize=9)
axes[2].set_xlabel('Parámetros (M)', fontsize=10)
axes[2].set_ylabel('Test F1-macro', fontsize=10)
axes[2].set_title('Eficiencia: F1 vs Parámetros')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_global_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_global_metrics.png')

## Sección 7 — Matrices de confusión

Matrices de confusión normalizadas por fila (recall por clase) para los cuatro modelos.  
La diagonal principal representa la tasa de clasificación correcta (recall por clase).


In [ ]:
n_models         = len(results)
model_names_list = list(results.keys())

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
axes = axes.flatten()

for idx, mname in enumerate(model_names_list):
    r  = results[mname]
    ax = axes[idx]
    sns.heatmap(
        r['cm_norm'],
        annot=True, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_ORDER,
        yticklabels=CLASS_ORDER,
        ax=ax, vmin=0, vmax=1,
        linewidths=0.3, linecolor='white',
    )
    ax.set_title(
        f'{r["label"]}  —  Test F1 = {r["f1_macro"]:.4f}  |  Acc = {r["accuracy"]:.4f}',
        fontsize=10, fontweight='bold', pad=8,
    )
    ax.set_xlabel('Predicho', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)
    ax.set_xticklabels(CLASS_ORDER, rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels(CLASS_ORDER, rotation=0, fontsize=8)

# Ocultar subplots vacíos si hay menos de 4 modelos evaluados
for idx in range(n_models, 4):
    axes[idx].set_visible(False)

plt.suptitle('Matrices de confusión normalizadas — Test set',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_confusion_matrices.png')

## Sección 8 — Rendimiento por clase

Comparación del F1-score y recall por clase para las cuatro arquitecturas.  
Permite identificar qué clases mejoran con cada modelo y dónde existen límites compartidos.


In [ ]:
# F1 por clase × modelo
per_class_f1 = {
    r['label']: [r['report'][cls]['f1-score'] for cls in CLASS_ORDER]
    for r in results.values()
}

x   = np.arange(len(CLASS_ORDER))
n_m = len(per_class_f1)
w   = 0.7 / n_m

fig, ax = plt.subplots(figsize=(14, 6))
for i, (lbl, f1_cls) in enumerate(per_class_f1.items()):
    offset = (i - n_m / 2 + 0.5) * w
    color  = next(r['color'] for r in results.values() if r['label'] == lbl)
    ax.bar(x + offset, f1_cls, width=w * 0.9,
           label=lbl, color=color, edgecolor='white', linewidth=0.5, alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_ORDER, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1-score', fontsize=11)
ax.set_title('F1-score por clase — Comparativa de modelos (test set)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_per_class_f1.png')

In [ ]:
# Heatmap de recall por clase
recall_matrix = {
    r['label']: [r['report'][cls]['recall'] for cls in CLASS_ORDER]
    for r in results.values()
}
recall_df = pd.DataFrame(recall_matrix, index=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    recall_df,
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0.3, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax,
)
ax.set_title('Recall por clase — Comparativa de modelos (test set)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_yticklabels(CLASS_ORDER, rotation=0, fontsize=10)
ax.set_xticklabels(list(recall_matrix.keys()), rotation=15, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_recall_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_recall_heatmap.png')

print('\n── Recall por clase ──────────────────────────────────────')
print(recall_df.to_string(float_format=lambda x: f'{x:.3f}'))

## Sección 9 — Curvas de entrenamiento comparadas

Evolución del val F1-macro durante el entrenamiento para las cuatro arquitecturas,  
cargadas desde los archivos de log CSV generados por cada notebook de entrenamiento.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Curvas de entrenamiento comparadas', fontsize=13, fontweight='bold')

for mname, cfg in MODEL_CONFIGS.items():
    log_path = LOG_DIR / f'{mname}_log.csv'
    if not log_path.exists():
        print(f'  [SKIP] {cfg["label"]} — {log_path.name} no encontrado')
        continue

    df    = pd.read_csv(log_path)
    label = cfg['label']
    color = cfg['color']

    # Panel izquierdo: solo val F1
    axes[0].plot(df['epoch'], df['val_f1'], label=label, color=color, linewidth=1.8)

    # Marcar el mejor checkpoint
    best_ep = df.loc[df['val_f1'].idxmax(), 'epoch']
    best_f1 = df['val_f1'].max()
    axes[0].scatter(best_ep, best_f1, color=color, s=70, zorder=5,
                    edgecolors='white', linewidths=0.8)

    # Panel derecho: train (--) vs val (—)
    axes[1].plot(df['epoch'], df['val_f1'],   color=color, linewidth=1.8)
    axes[1].plot(df['epoch'], df['train_f1'], color=color, linewidth=1.2,
                 linestyle='--', alpha=0.6)

axes[0].set_xlabel('Epoch', fontsize=10)
axes[0].set_ylabel('Val F1-macro', fontsize=10)
axes[0].set_title('Val F1-macro (todos los modelos)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch', fontsize=10)
axes[1].set_ylabel('F1-macro', fontsize=10)
axes[1].set_title('Train (- -) vs Val (—) F1-macro')
axes[1].grid(alpha=0.3)

# Leyenda manual para el panel derecho
patches = [
    mpatches.Patch(color=cfg['color'], label=cfg['label'])
    for mname, cfg in MODEL_CONFIGS.items()
    if (LOG_DIR / f'{mname}_log.csv').exists()
]
axes[1].legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_training_curves.png')

## Sección 10 — Análisis de eficiencia

Comparación de la eficiencia de cada modelo: rendimiento (test F1-macro) frente a  
coste computacional (parámetros totales, tiempo total de entrenamiento efectivo).


In [ ]:
# Tiempo total de entrenamiento desde los logs CSV (suma de elapsed_s)
train_times_h = {}
for mname, cfg in MODEL_CONFIGS.items():
    log_path = LOG_DIR / f'{mname}_log.csv'
    if log_path.exists():
        df = pd.read_csv(log_path)
        train_times_h[cfg['label']] = df['elapsed_s'].sum() / 3600

# ─── Tabla de eficiencia ─────────────────────────────────────────────────────
print('── Análisis de eficiencia ──────────────────────────────────────')
header = f'{"Modelo":<20} {"Test F1":>9} {"Params (M)":>12} {"F1/Param×100":>14} {"T train (h)":>13}'
print(header)
print('-' * len(header))
for mname, r in results.items():
    lbl   = r['label']
    f1    = r['f1_macro']
    p     = r['params_m']
    ratio = (f1 / p) * 100
    t_str = f'{train_times_h[lbl]:.1f}' if lbl in train_times_h else 'N/A'
    print(f'{lbl:<20} {f1:>9.4f} {p:>12.1f} {ratio:>14.4f} {t_str:>13}')

# ─── Scatter plots ───────────────────────────────────────────────────────────
model_labels_eff = [r['label']    for r in results.values()]
f1_eff           = [r['f1_macro'] for r in results.values()]
params_eff       = [r['params_m'] for r in results.values()]
colors_eff       = [r['color']    for r in results.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis de eficiencia — Test F1-macro', fontsize=12, fontweight='bold')

# F1 vs parámetros
axes[0].scatter(params_eff, f1_eff, c=colors_eff, s=200, zorder=3,
                edgecolors='white', linewidths=1.2)
for lbl, px, fy in zip(model_labels_eff, params_eff, f1_eff):
    axes[0].annotate(lbl, (px, fy), textcoords='offset points', xytext=(6, 4), fontsize=9)
axes[0].set_xlabel('Parámetros (M)', fontsize=10)
axes[0].set_ylabel('Test F1-macro', fontsize=10)
axes[0].set_title('F1-macro vs Parámetros')
axes[0].grid(alpha=0.3)

# F1 vs tiempo de entrenamiento
time_data = [
    (train_times_h.get(r['label']), r['f1_macro'], r['label'], r['color'])
    for r in results.values()
    if r['label'] in train_times_h
]
if time_data:
    t_v, f_v, l_v, c_v = zip(*time_data)
    axes[1].scatter(t_v, f_v, c=c_v, s=200, zorder=3, edgecolors='white', linewidths=1.2)
    for lbl, tx, fy in zip(l_v, t_v, f_v):
        axes[1].annotate(lbl, (tx, fy), textcoords='offset points', xytext=(6, 4), fontsize=9)
    axes[1].set_xlabel('Tiempo de entrenamiento (h)', fontsize=10)
    axes[1].set_ylabel('Test F1-macro', fontsize=10)
    axes[1].set_title('F1-macro vs Tiempo de entrenamiento')
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Logs de entrenamiento no disponibles',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=10)

plt.tight_layout()
plt.savefig(LOG_DIR / 'evaluation_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {LOG_DIR}/evaluation_efficiency.png')

# ─── Guardar tabla resumen ───────────────────────────────────────────────────
summary_df.to_csv(LOG_DIR / 'evaluation_summary.csv', index=False)
print(f'Tabla resumen guardada: {LOG_DIR}/evaluation_summary.csv')

## Sección 11 — Resumen

**Artefactos generados:**
- `logs/evaluation_global_metrics.png` — F1-macro, accuracy y F1 vs parámetros
- `logs/evaluation_confusion_matrices.png` — matrices de confusión normalizadas (2×2 grid)
- `logs/evaluation_per_class_f1.png` — F1-score por clase (barras agrupadas)
- `logs/evaluation_recall_heatmap.png` — recall por clase (heatmap, verde/rojo)
- `logs/evaluation_training_curves.png` — curvas de val F1 comparadas (train vs val)
- `logs/evaluation_efficiency.png` — F1 vs parámetros y tiempo de entrenamiento
- `logs/evaluation_summary.csv` — tabla resumen con todas las métricas

**Conclusiones del pipeline:**
- **Mejor test F1:** Swin-S (~0.6962) — seguido por MaxViT-T (~0.6951), ResNet-50 (~0.6914), EfficientNet-B3 (~0.6894)
- **Mejor eficiencia (F1/params):** EfficientNet-B3 — mayor F1 por millón de parámetros (~10.7 M)
- **Mejora marginal por capacidad:** pasar de 10.7 M (EfficientNet-B3) a 49.6 M (Swin-S) solo mejora +0.0068 F1
- **Clase más fácil:** Edge_on — recall ≥ 0.91 en todos los modelos
- **Clase más difícil:** Lenticular — recall ≤ 0.51 en todos los modelos; límite inherente a la ambigüedad E/S0
- **Límite fundamental:** la distinción Elliptical ↔ Lenticular no mejora con mayor capacidad de modelo

**Trabajo futuro:**
- Entrenar Swin-T y añadirlo a esta comparativa
- Aumentar la resolución de entrada (≥ 448 px) para capturar más detalle morfológico
- Aplicar MixUp / Label Smoothing para regularizar fronteras de decisión ambiguas
- Explorar ensamble de modelos para mejorar Lenticular e Irregular
